# SHARP-LLM CodeT5-Small Convergence Experiment
## Kaggle Kernel - Automatic Setup and Upload

This notebook demonstrates how to create, validate, and push a Kaggle kernel programmatically for the convergence experiment.

## Step 1: Install and Verify Kaggle CLI

In [ ]:
import subprocess
import sys

# Install/upgrade Kaggle CLI
print("Installing Kaggle CLI...")
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'kaggle', '-q'])
print("✓ Kaggle CLI installed")

# Verify installation
result = subprocess.run(['kaggle', '--version'], capture_output=True, text=True)
print(f"Kaggle version: {result.stdout.strip()}")
if result.returncode != 0:
    print(f"Warning: {result.stderr}")
else:
    print("✓ Kaggle CLI verified")

## Step 2: Authenticate with kaggle.json

Ensure kaggle.json is in ~/.kaggle/ directory (or %USERPROFILE%\.kaggle\ on Windows)

In [ ]:
import os
import json
from pathlib import Path

# Locate kaggle.json
kaggle_dir = Path.home() / '.kaggle'
kaggle_json = kaggle_dir / 'kaggle.json'

print(f"Kaggle config dir: {kaggle_dir}")
print(f"Kaggle config file: {kaggle_json}")

if kaggle_json.exists():
    print(f"✓ Found {kaggle_json}")
    with open(kaggle_json) as f:
        creds = json.load(f)
        print(f"  Username: {creds.get('username')}")
        print(f"  API key configured: Yes")
else:
    print(f"✗ NOT FOUND: {kaggle_json}")
    print("\n📌 To authenticate:")
    print(f"  1. Download kaggle.json from https://www.kaggle.com/settings/account")
    print(f"  2. Save it to: {kaggle_json}")
    print(f"  3. Run: chmod 600 {kaggle_json}  (on Linux/Mac)")

# Test authentication
print("\nTesting Kaggle API authentication...")
result = subprocess.run(['kaggle', 'api', 'version', '--help'], capture_output=True, text=True)
if result.returncode == 0:
    print("✓ Kaggle authentication working")
else:
    print(f"✗ Authentication failed: {result.stderr[:100]}")

## Step 3: Create and Configure Kernel Folder for Kaggle

Prepare the folder and create minimal kernel metadata

In [ ]:
# Setup kernel folder
KERNEL_FOLDER = Path('kaggle_sharp_llm_convergence')
KERNEL_FOLDER.mkdir(exist_ok=True)

print(f"Kernel folder: {KERNEL_FOLDER.absolute()}")

# Create kernel metadata JSON (run this to generate kernel-metadata.json)
print("\n✓ Kernel folder ready")
print("  Next: Run `kaggle kernels init -p kaggle_sharp_llm_convergence`")
print("  Or continue to next cell to create metadata programmatically")

## Step 4: Initialize Kaggle Kernel Metadata

Create kernel-metadata.json with proper configuration

In [ ]:
# Create kernel-metadata.json programmatically
kernel_metadata = {
    "id": "msbasanth/sharp-llm-convergence-codet5",
    "title": "SHARP-LLM CodeT5-Small Convergence Experiment",
    "code_file": "sharp_llm_convergence_kaggle.ipynb",
    "language": "python",
    "kernel_type": "notebook",
    "is_private": True,
    "enable_gpu": True,
    "enable_internet": True,
    "dataset_sources": [],
    "competition_sources": [],
    "kernel_sources": []
}

metadata_path = KERNEL_FOLDER / 'kernel-metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(kernel_metadata, f, indent=2)

print(f"✓ Created kernel metadata: {metadata_path}")
print(json.dumps(kernel_metadata, indent=2))

## Step 5: Copy Notebook and Validate Files

In [ ]:
import shutil

# Copy this notebook to the kernel folder
notebook_src = Path('sharp_llm_convergence_kaggle.ipynb')
notebook_dst = KERNEL_FOLDER / 'sharp_llm_convergence_kaggle.ipynb'

if notebook_src.exists():
    shutil.copy(str(notebook_src), str(notebook_dst))
    print(f"✓ Copied notebook: {notebook_dst}")
else:
    print(f"⚠ Notebook not found at {notebook_src} (will be created on Kaggle)")

# Validate files before push
print("\n✓ Preflight Check:")
print(f"  Kernel folder: {KERNEL_FOLDER.absolute()}")
print(f"  Files in folder:")
for f in KERNEL_FOLDER.iterdir():
    size = f.stat().st_size / 1024  # KB
    print(f"    - {f.name} ({size:.1f} KB)")

# Load and display metadata
with open(metadata_path) as f:
    meta = json.load(f)
    print(f"\n✓ Kernel metadata:")
    print(f"    ID: {meta['id']}")
    print(f"    Title: {meta['title']}")
    print(f"    GPU: {meta.get('enable_gpu', False)}")
    print(f"    Private: {meta.get('is_private', True)}")

## Step 6: Push Kernel to Kaggle

Upload the notebook and metadata to Kaggle

In [ ]:
print("Pushing kernel to Kaggle...")
print(f"Command: kaggle kernels push -p {KERNEL_FOLDER}")

result = subprocess.run(
    ['kaggle', 'kernels', 'push', '-p', str(KERNEL_FOLDER)],
    capture_output=True,
    text=True
)

print("STDOUT:")
print(result.stdout)

if result.returncode == 0:
    print("\n✓ Kernel pushed successfully!")
    kernel_id = kernel_metadata['id']
    print(f"  View at: https://www.kaggle.com/{kernel_id}")
else:
    print("\n✗ Push failed!")
    print("STDERR:")
    print(result.stderr)
    print("\nTroubleshooting:")
    print("  1. Verify kaggle.json is in ~/.kaggle/")
    print("  2. Check internet connection")
    print("  3. Ensure Kaggle username matches in kernel-metadata.json")

## Step 7: Monitor Kernel Status

Check execution status and view outputs

In [ ]:
import time

kernel_id = kernel_metadata['id']
print(f"Checking status of: {kernel_id}\n")

# Monitor kernel status
max_checks = 5
check_interval = 10  # seconds

for i in range(max_checks):
    result = subprocess.run(
        ['kaggle', 'kernels', 'status', kernel_id],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        status = result.stdout.strip()
        print(f"[{i+1}/{max_checks}] Status: {status}")
        
        if 'running' in status.lower():
            print(f"  ⏳ Kernel is running (expected ~2-3 hours on T4)...")
        elif 'complete' in status.lower():
            print(f"  ✓ Kernel completed!")
            break
    else:
        print(f"[{i+1}/{max_checks}] Could not get status: {result.stderr[:100]}")
    
    if i < max_checks - 1:
        print(f"  Checking again in {check_interval}s...\n")
        time.sleep(check_interval)

print(f"\n📊 View full kernel output:")
print(f"  https://www.kaggle.com/{kernel_id}")
print(f"\n💾 Download outputs after completion from kernel page or via:")
print(f"  kaggle kernels output {kernel_id}")